# DataPulse MY — Trust Layer in a Notebook

Learn the sequence: **check health → inspect evidence → decide whether to use → fetch official data**.

## Safeguards

1. **Verification-first artifact rule.** Every recipe, notebook, case study, or badge must show the dataset ID, canonical official source, DataPulse status, `checked_at`, freshness evidence, and any caveat before showing the data result. An artifact missing any field does not publish.

2. **Fail closed in examples.** Examples may proceed automatically only for statuses explicitly allowed by the recipe. `stale`, `degraded`, `unreachable`, `unknown`, and `unknown-freshness` must produce a visible stop or warning; `browser-dependent` and `Volatile` must explain their limitations. No status may be collapsed into a generic green “available” label.

3. **Trust-first information architecture.** The homepage hero and primary CTA remain “verified health” and “view live status/connect your agent.” Outreach artifacts live under a secondary “Learn by example” path and may not replace the live status distribution, taxonomy, or verification mechanism above the fold.

4. **Contribution boundary.** Community work is accepted only if it improves at least one recorded trust metric: verified coverage, content-date extraction, schema-drift detection, probe reliability, provenance, licence evidence, or reproducible failure documentation. Requests for charts, data cleaning, hosted querying, alerts, or unrelated APIs are declined or logged as paid-layer discovery.

5. **Maintainer-controlled evidence.** No community submission may directly alter production probe policy or status classification. It requires review, a reproducible fixture, source evidence, and a successful canary run. Recognition is granted only after acceptance.

6. **Canonical-source rule.** All artifacts consume the same published health envelope and taxonomy; no notebook, badge, or page maintains its own status logic. A taxonomy change must update the schema and all affected artifacts together.

7. **No free convenience creep.** The outreach layer may teach how to check trust and then access an official source. It may not host normalized datasets, promise stable schemas, run persistent alerts, provide cross-source joins, or guarantee delivery. Those are convenience-layer capabilities and require an explicit product decision.

8. **Evidence over audience metrics.** The 30-day scorecard contains: external citations/backlinks to a reproducible artifact, successful notebook runs, MCP queries that include health/provenance, accepted probe improvements, and unknown-freshness cases resolved. Followers, page views, submissions, and demo count are diagnostic only and cannot justify more scope.

9. **Fixed outreach budget.** Until a paid convenience offer is validated, outreach maintenance is capped at one maintainer-day per week and may add no standalone database, queue, account system, or always-on service. Work exceeding either limit pauses for a layer-boundary review.

10. **ODIN claim discipline.** Every use of the ranking must say what it measures—coverage/openness—and must not imply that ODIN certifies freshness, DataPulse, or individual datasets. DataPulse is an independent verification layer, not an official ODIN or Malaysian-government endorsement.

11. **Public limitation statement.** Each artifact must state that DataPulse verifies defined observable signals, not the semantic truth or fitness-for-purpose of every record. This prevents an honest health status from being read as a blanket data-quality guarantee.

12. **One-way escalation rule.** Repeated requests for wrappers, cleaned feeds, stable schemas, or alerts are interviewed as potential paid convenience demand; they are not added to the free trust layer. Build only after a named buyer, use case, and willingness to pay are established.

**What this notebook proves:** verified MY open data is usable right now, and the trust surface shows you exactly when it isn't.

## Why this matters

Malaysia ranks #1 in the world for open data coverage and openness (ODIN 2024/25, 99/100 for openness). But "published" ≠ "fresh" — most endpoints don't send Last-Modified headers, and 200 OK can hide years of staleness. ODIN does not certify freshness, DataPulse, or individual datasets.

DataPulse independently probes **335 official datasets every 15 minutes** and classifies each into one of 8 honest statuses. This notebook shows you how to read that surface and decide which datasets to actually use.

In [ ]:
import requests, pandas as pd
HEALTH_URL = "https://data-pulse.my/health/latest.json"
health = requests.get(HEALTH_URL, timeout=10).json()
ts = health["_trust_summary"]
ds = health["datasets"]
print(f"Total datasets: {ts['datasets_total']}")
print(f"Checked at:    {ts['checked_at']}")
print(f"Status distribution: {ts['by_status']}")

In [ ]:
# Render the 9-status taxonomy
status_df = pd.DataFrame([
    {"status": k, "count": v, "percent": f"{v/ts['datasets_total']*100:.1f}%"}
    for k, v in sorted(ts['by_status'].items(), key=lambda kv: -kv[1])
])
status_df.style.hide(axis='index').format({'count': '{:,}'})

## Example 1: a fresh dataset you can use right now

Use `fuelprice` — daily fuel prices from Malaysia's Ministry of Finance. The live evidence below reports when it was last probed, its content freshness date, official source, and current status. Because status can change, the next cell refuses to fetch unless the live status is `fresh` or `aging`.

In [ ]:
fresh_example = "fuelprice"
fresh_rec = next(d for d in ds if d["dataset_id"] == fresh_example)
print(f"Dataset:     {fresh_rec['dataset_id']}")
print(f"Status:      {fresh_rec['status']}")
print(f"Last check:  {fresh_rec['last_checked']}")
print(f"Fresh date:  {fresh_rec['content_freshness_date']}")
print(f"Rows:        {fresh_rec['record_count']:,}" if isinstance(fresh_rec['record_count'], int) else f"Rows: {fresh_rec['record_count']}")
print(f"Source URL:  {fresh_rec['request_url']}")
print("Caveat:      Status verifies observable freshness signals, not every value.")

In [ ]:
# Verification-first: refuse to use this dataset unless status is acceptable
ALLOWED = {"fresh", "aging"}
if fresh_rec["status"] not in ALLOWED:
    raise SystemExit(f"BLOCKED: {fresh_example} status is {fresh_rec['status']}; not safe to fetch")

print(f"✓ Status '{fresh_rec['status']}' is allowed. Fetching official data...")
source_response = requests.get(fresh_rec["request_url"], timeout=30)
source_response.raise_for_status()
df = pd.DataFrame(source_response.json())
print(f"Fetched {len(df):,} rows × {len(df.columns)} columns")
df.head()

## Example 2: a dataset you should NOT use right now

Find the oldest stale dataset by its live `staleness_days` evidence. Show its dataset ID, status, last check, freshness evidence, official source, and caveat. Refuse to fetch.

In [ ]:
def require_current_dataset(record):
    blocked = {"stale", "degraded", "unreachable", "unknown", "unknown_freshness"}
    if record["status"] in blocked:
        raise SystemExit(f"BLOCKED: '{record['status']}' status; do NOT use this dataset for current decisions.")

stale_examples = [d for d in ds if d["status"] == "stale"]
oldest = max(stale_examples, key=lambda d: (d.get("staleness_days") or -1, d["dataset_id"]))
print(f"Dataset:    {oldest['dataset_id']}")
print(f"Status:     {oldest['status']}")
print(f"Last check: {oldest['last_checked']}")
print(f"Fresh date: {oldest.get('content_freshness_date', 'N/A')}")
print(f"Last mod:   {oldest.get('last_modified', 'N/A')}")
print(f"Days old:   {oldest.get('staleness_days', 'N/A')}")
print(f"Source URL: {oldest['request_url']}")
print("Caveat:     Suitable only for explicitly dated historical research.")
print()

# Fail-closed: expose the stop visibly without fetching stale data.
try:
    require_current_dataset(oldest)
except SystemExit as error:
    print(f"⛔ {error}")
    evidence_date = oldest.get('content_freshness_date') or oldest.get('last_modified', 'unknown')
    print(f"   For historical research, cite explicitly: \"as of {evidence_date}\"")

## Bonus: find 10 datasets you can use right now (fresh or aging only)

In [ ]:
usable = [d for d in ds if d["status"] in ("fresh", "aging")]
print(f"Usable right now: {len(usable)} of {ts['datasets_total']} ({len(usable)/ts['datasets_total']*100:.1f}%)")
print(f"\nFirst 10 (sorted by most-recently-checked):")
sorted_usable = sorted(usable, key=lambda d: d["last_checked"], reverse=True)
for d in sorted_usable[:10]:
    print(f"  {d['dataset_id']:35}  status={d['status']:8}  last_checked={d['last_checked']}")

## Limitations

DataPulse verifies defined observable signals (HTTP status, header freshness, content-date extraction, record-count stability, schema shape). It does NOT verify the semantic truth or fitness-for-purpose of every record. A fresh dataset is one you can use **knowing the publisher's update cadence is being honored**, not one whose every value is correct.

**Cite this notebook:**
> DataPulse MY. (2026). *Trust Layer in a Notebook*. Retrieved from
> https://data-pulse.my/docs/trust-layer-notebook.ipynb